<a href="https://colab.research.google.com/github/ivywanjage/cdav/blob/main/kenya_transactions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [82]:
import pandas as pd
pd.__version__

'2.2.2'

In [83]:
df=pd.read_csv("/content/kenya_transactions_messy.csv")
df.head()

,Txn ID,Full Name,County,Amount Paid,Txn Date,Paid Date,Rainfall mm,Status
0,TXN00729,"Wanjiku, Jane",NAIROBI,76,22/02/2026,12/03/2026,222.9,?
1,TXN00798,"Onyango, Victor",Kisumu,-,03/06/2026,NaN,902.6,P
2,TXN00672,"Muthoni, John",Kiambu,6920,26/04/2026,11/05/2026,1122.0,P
3,TXN00660,"Barasa, David",Kajiado,4342,21/02/2026,05/03/2026,509.3,P
4,TXN00830,"Kimani, Michael",Kisumu,452,11/03/2026,10/04/2026,1193.9,P


In [84]:
df.shape

(1000, 8)

In [85]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Txn ID         1000 non-null   object 
 1   Full Name      1000 non-null   object 
 2   County         993 non-null    object 
 3    Amount Paid   960 non-null    object 
 4   Txn Date       1000 non-null   object 
 5   Paid Date      849 non-null    object 
 6   Rainfall mm    963 non-null    float64
 7   Status         879 non-null    object 
dtypes: float64(1), object(7)
memory usage: 62.6+ KB


In [86]:
df.columns

Index(['Txn ID', 'Full Name', 'County ', ' Amount Paid ', 'Txn Date',
       'Paid Date', 'Rainfall mm', 'Status'],
      dtype='object')

In [87]:
df.describe(include="all")

,Txn ID,Full Name,County,Amount Paid,Txn Date,Paid Date,Rainfall mm,Status
count,1000,1000,993,960,1000,849,963.000000,879
unique,945,663,44,818,184,199,NaN,3
top,TXN00830,"Kimani, Ann",Mombasa,-,16/02/2026,06/05/2026,NaN,P
freq,3,6,119,20,12,12,NaN,438
mean,NaN,NaN,NaN,NaN,NaN,NaN,956.926376,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,439.099355,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,180.800000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,585.650000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,952.400000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,1316.050000,NaN


In [88]:
missing=pd.DataFrame({
    'missing_count':df.isnull().sum(),
    'missing_pct':(df.isnull().sum()/len(df)*100).round(1)

})
missing

,missing_count,missing_pct
Txn ID,0,0.0
Full Name,0,0.0
County,7,0.7
Amount Paid,40,4.0
Txn Date,0,0.0
Paid Date,151,15.1
Rainfall mm,37,3.7
Status,121,12.1


In [89]:
#before-note the spaces around some names
print('before',df.columns.tolist())

before ['Txn ID', 'Full Name', 'County ', ' Amount Paid ', 'Txn Date', 'Paid Date', 'Rainfall mm', 'Status']


In [90]:
df.columns=(
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ','_')
)

In [91]:
print("After:", df.columns.tolist())

After: ['txn_id', 'full_name', 'county', 'amount_paid', 'txn_date', 'paid_date', 'rainfall_mm', 'status']


In [92]:
print('Total rows:      ',len(df))
print('Duplicate txn_id rows',df.duplicated('txn_id').sum())

Total rows:       1000
Duplicate txn_id rows 55


In [100]:
#show both copies of a duplicated transaction
example_id=df[df.duplicated('txn_id', keep=False)]['txn_id'].iloc[0]
df[df['txn_id']==example_id]

,txn_id,full_name,county,amount_paid,txn_date,paid_date,rainfall_mm,status
4,TXN00830,"Kimani, Michael",Kisumu,452,11/03/2026,10/04/2026,1193.9,Paid
417,TXN00830,"Kimani, Michael",Kisumu,2757,11/03/2026,14/03/2026,1193.9,Paid


In [101]:
#keep 'first' retains the first occurence and drops the rest
#reset_index(drop=True) renumbers the rows from 0 after the drop
df=df.drop_duplicates(subset='txn_id',keep='first').reset_index(drop=True)

In [105]:
#verify
print('rows after duplication: ',len(df))
print('duplicate the ids in remaining: ',df.duplicated('txn_id').sum())

rows after duplication:  945
duplicate the ids in remaining:  0


In [107]:
print('Unique counties before cleaning:',df['county'].nunique())
df['county'].value_counts().head(10)

Unique counties before cleaning: 45


,count
county,
Mombasa,115
Nairobi,98
Kisumu,75
Nakuru,62
Eldoret,54
Kiambu,53
Machakos,42
Meru,39
NAIROBI,38


In [109]:
#fix
#chain two .str methods:.str.strip(),.str.title()
df['county']=df['county'].str.strip().str.title()

In [111]:
#verify
print('unique counties after cleaning:',df['county'].nunique())
df['county'].value_counts()

unique counties after cleaning: 16


,count
county,
Nairobi,192
Mombasa,126
Kisumu,87
Nakuru,72
Eldoret,66
Kiambu,59
Machakos,47
Meru,42
Nyeri,41


In [113]:
#clean full_name column
mask_double=df['full_name'].str.contains(', ',na=False)
print('names with extra space after comma:',mask_double.sum())
df.loc[mask_double,'full_name'].head()

names with extra space after comma: 945


,full_name
0,"Wanjiku, Jane"
1,"Onyango, Victor"
2,"Muthoni, John"
3,"Barasa, David"
4,"Kimani, Michael"


In [ ]:
#step 1: collapse any run of spaces after the comma to a single space
df['full_name']=df['full_name'].str.replace(r',\s+', ', ',regex=True)

#step 2:split into surname and first name
df[['surname','first_name']]=(
    df['full_name']
      .str.split(',', n=1,expand=True)
      .apply(lambda)
)

In [ ]:
print('no more double space?:')

In [93]:
df.isna()

,txn_id,full_name,county,amount_paid,txn_date,paid_date,rainfall_mm,status
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,True,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
995,False,False,False,False,False,False,False,True
996,False,False,False,False,False,True,False,False
997,False,False,False,False,False,False,False,False
998,False,False,False,False,False,False,False,True


In [94]:
df.duplicated().sum()

np.int64(40)

In [95]:
df[df.duplicated()]

,txn_id,full_name,county,amount_paid,txn_date,paid_date,rainfall_mm,status
161,TXN00750,"Wafula, Mercy",Nakuru,-260,28/01/2026,05/02/2026,1334.8,?
235,TXN00562,"Waweru, Samuel",Kiambu,754,23/06/2026,NaN,NaN,U
263,TXN00855,"Muthoni, Samuel",Kitale,496,04/01/2026,30/01/2026,1747.7,P
286,TXN00001,"Kiptoo, Samuel",Kiambu,608,24/01/2026,NaN,973.4,P
314,TXN00840,"Korir, Alice",Kericho,3805,22/05/2026,26/05/2026,412.8,P
320,TXN00734,"Omondi, Grace",nairobi,4507 KES,05/02/2026,07/03/2026,586.7,P
423,TXN00148,"Muthoni, Paul",nairobi,3042,18/02/2026,21/02/2026,1314.2,NaN
445,TXN00772,"Ochieng, James",BUNGOMA,"3,193 KES",18/05/2026,05/06/2026,781.2,U
473,TXN00660,"Barasa, David",Kajiado,4342,21/02/2026,05/03/2026,509.3,P
497,TXN00418,"Ochieng, Dennis",Kakamega,-,28/05/2026,12/06/2026,1482.4,U


In [96]:
df = df.drop_duplicates()
display(df.head())

,txn_id,full_name,county,amount_paid,txn_date,paid_date,rainfall_mm,status
0,TXN00729,"Wanjiku, Jane",NAIROBI,76,22/02/2026,12/03/2026,222.9,?
1,TXN00798,"Onyango, Victor",Kisumu,-,03/06/2026,NaN,902.6,P
2,TXN00672,"Muthoni, John",Kiambu,6920,26/04/2026,11/05/2026,1122.0,P
3,TXN00660,"Barasa, David",Kajiado,4342,21/02/2026,05/03/2026,509.3,P
4,TXN00830,"Kimani, Michael",Kisumu,452,11/03/2026,10/04/2026,1193.9,P


In [97]:
df = df.fillna(0)

In [98]:
df["status"]=df["status"].replace({
    "P":"Paid","U":"Unpaid","?":"None"
    })

In [99]:
df.dropna()

,txn_id,full_name,county,amount_paid,txn_date,paid_date,rainfall_mm,status
0,TXN00729,"Wanjiku, Jane",NAIROBI,76,22/02/2026,12/03/2026,222.9,None
1,TXN00798,"Onyango, Victor",Kisumu,-,03/06/2026,0,902.6,Paid
2,TXN00672,"Muthoni, John",Kiambu,6920,26/04/2026,11/05/2026,1122.0,Paid
3,TXN00660,"Barasa, David",Kajiado,4342,21/02/2026,05/03/2026,509.3,Paid
4,TXN00830,"Kimani, Michael",Kisumu,452,11/03/2026,10/04/2026,1193.9,Paid
...,...,...,...,...,...,...,...,...
994,TXN00774,"Mutua, Ruth",Nairobi,562,05/01/2026,17/01/2026,737.3,0
995,TXN00248,"Rotich, Stephen",Nairobi,11105,06/06/2026,18/06/2026,896.8,0
996,TXN00314,"Auma, Samuel",Mombasa,17377,17/01/2026,0,639.9,Paid
998,TXN00712,"Cheruiyot, Jane",Kericho,4862,09/02/2026,14/02/2026,1636.0,0
